In [62]:
from PIL import Image
import os
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import sklearn.metrics

import xml.etree.ElementTree as ET

In [63]:
from train import EmbeddingNet

In [64]:
class VeRiEval(torch.utils.data.Dataset):
    def __init__(self, data_dir, file, transform=None):
        self.root = data_dir
        self.transform = transform or transforms.ToTensor()

        with open(file, 'r') as f:
            l = f.readlines()

        self.img_names = [line.strip('\n') for line in l]

    def __len__(self):
        return len(self.img_names)
    
    def __getitem__(self, idx):
        img_name = self.img_names[idx]

        # parse filename like "0002_c002_00030600_0.jpg"
        parts = img_name.split('_')
        vid = int(parts[0])             # "0002" -> 2
        cid = int(parts[1][1:])         # "c002" -> 2 (skip 'c')

        img = Image.open(os.path.join(self.root, img_name)).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, vid, cid

In [65]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EmbeddingNet()
model.to(device)
model.load_state_dict(torch.load("../server/veri_emb_rn50.pt", map_location=device))

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                            std=[0.229, 0.224, 0.225]),
])

C:\Users\Makai\AppData\Local\Temp\ipykernel_45852\4002849990.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../server/veri_emb_rn50.pt

In [66]:
query = VeRiEval(data_dir='../datasets/VeRi/image_query/', file='../datasets/VeRi/name_query.txt', transform=transform)
query_dl = torch.utils.data.DataLoader(query, batch_size=256)

gallery = VeRiEval(data_dir='../datasets/VeRi/image_test/', file='../datasets/VeRi/name_test.txt', transform=transform)
gallery_dl = torch.utils.data.DataLoader(gallery, batch_size=256)

In [67]:
q_embs, q_vids, q_cids = [], [], [] # queries
g_embs, g_vids, g_cids = [], [], [] # gallery

model.eval()

with torch.no_grad():
    for imgs, vids, cids in query_dl:
        imgs = imgs.to(device)
        emb = model(imgs).flatten(1)
        emb = F.normalize(emb, p=2, dim=1)
        
        q_embs.append(emb.cpu())
        q_vids.extend(vids.numpy().astype(int).tolist())
        q_cids.extend(cids.numpy().astype(int).tolist())

    for imgs, vids, cids in gallery_dl:
        imgs = imgs.to(device)
        emb = model(imgs).flatten(1)
        emb = F.normalize(emb, p=2, dim=1)
        
        g_embs.append(emb.cpu())
        g_vids.extend(vids.numpy().astype(int).tolist())
        g_cids.extend(cids.numpy().astype(int).tolist())

q_embs = torch.cat(q_embs, dim=0)
g_embs = torch.cat(g_embs, dim=0)

q_vids = np.array(q_vids)
q_cids = np.array(q_cids)

g_vids = np.array(g_vids)
g_cids = np.array(g_cids)

In [83]:
topk=(1,5,10)       # the top K for CMC curve
q_chunk = 128       # query batch size

q_num = q_embs.shape[0]
g_num = g_embs.shape[0]
cmc_counts = np.zeros(max(topk), dtype=int)
APs = []
val_q = 0

q_embs = q_embs.to(device)
g_embs = g_embs.to(device)

formulas

- $P = \frac{TP}{TP + FP}$

- $ mAP = \frac{1}{|\text{C}|} \sum_{c \ \in \ \text{C}} \frac{|TP_c|}{|FP_c| + |TP_c|} $

- $ CMC = P(Rank \leq k) = \frac{1}{N} \sum_{i=1}^N \text{match}(i,k) $

In [89]:
for q_start in range(0, q_num, q_chunk):
    q_end = min(q_num, q_start + q_chunk)
    q_chunk_embs = q_embs[q_start:q_end]

    dist = torch.cdist(q_chunk_embs, g_embs, p=2).cpu().numpy()  # shape: (q_chunk, g_num)

    for i in range(q_end - q_start):
        q_idx = q_start + i     # index of query out of the set
        q_vid = q_vids[q_idx]   # index of query vehicle ID
        q_cid = q_cids[q_idx]   # index of query camera ID

        ord = np.argsort(dist[i])   # sort by dist to query (ascending)
        g_vids_ord = g_vids[ord]
        g_cids_ord = g_cids[ord]

        junk = (g_vids_ord == q_vid) & (g_cids_ord == q_cid)
        g_vids_f = g_vids_ord[~junk]

        matches = (g_vids_f == q_vid).astype(int)

        if matches.sum() == 0:
            continue  # no valid query, so skip 
        
        val_q += 1 # increase the conut of valid queries

        first_pos = np.where(matches == 1)[0][0]
        cmc_counts[first_pos:] += 1

        c = np.cumsum(matches)
        hits = np.where(matches == 1)[0]
        prec = [c[k] / (k + 1) for k in hits]
        APs.append(np.mean(prec))

if val_q == 0:
    mAP = 0.0
    cmc = {k: 0.0 for k in topk}
else:
    mAP = float(np.mean(APs))
    cmc = cmc_counts.astype(float) / val_q
    # compute CMC for each top K
    # cmc[r] is rank-(r+1) accuracy
    results = {k: float(cmc[k-1]) if k-1 < len(cmc) else 0.0 for k in topk}


In [90]:
print(f'mAP: {mAP:.4f}')
print('CMC:', results)

mAP: 0.2120
CMC: {1: 0.36889153754469606, 5: 0.5816448152562574, 10: 0.6722288438617402}
